# LAB ASSIGNMENT 1

In [1]:
!pip install annoy faiss-cpu hnswlib

import pandas as pd
import numpy as np
import time
import faiss
import hnswlib
from annoy import AnnoyIndex
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 6.7 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 31.6 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520033 sha256=bb92ccc8a3b6bc4733bb63a28c0709f08efd00b57372d19997ef96523a46171c
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679799 sha256=c23d2d47e1742ffaddf9931975984ae6d44de3eae625d7bbe13ba8ba0cb16112
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

In [2]:
def recall_at_k(I_pred, I_gt, k):
  correct = 0
  for i in range(len(I_pred)):
    correct += len(set(I_pred[i][:k]) & set(I_gt[i][:k]))
  return correct / (len(I_pred) * k)

In [3]:
def exact_nn(X, Q, k, metric):
  dim = X.shape[1]
  if metric == 'euclidean':
    index = faiss.IndexFlatL2(dim)
    index.add(X)
    _, I = index.search(Q, k)
  else:
    Xn, Qn = X.copy(), Q.copy()
    faiss.normalize_L2(Xn)
    faiss.normalize_L2(Qn)
    index = faiss.IndexFlatIP(dim)
    index.add(Xn)
    _, I = index.search(Qn, k)
  return I

In [4]:
def run_annoy(X, Q, gt, k, metric):
  dim = X.shape[1]
  annoy_metric = 'euclidean' if metric == 'euclidean' else 'angular'
  index = AnnoyIndex(dim, annoy_metric)

  start = time.time()
  for i in range(len(X)):
    index.add_item(i, X[i])
  index.build(10)  # 10 trees
  build_time = time.time() - start

  start = time.time()
  I = [index.get_nns_by_vector(q, k) for q in Q]
  query_ms = (time.time() - start) * 1000 / len(Q)

  return build_time, query_ms, recall_at_k(I, gt, k)

In [5]:
def run_faiss(X, Q, gt, k, metric):
  dim = X.shape[1]
  nlist = 10 if len(X) <= 1000 else 100

  Xf, Qf = X.copy(), Q.copy()
  if metric == 'euclidean':
    quantizer = faiss.IndexFlatL2(dim)
    mt = faiss.METRIC_L2
  else:
    faiss.normalize_L2(Xf)
    faiss.normalize_L2(Qf)
    quantizer = faiss.IndexFlatIP(dim)
    mt = faiss.METRIC_INNER_PRODUCT

  start = time.time()
  index = faiss.IndexIVFFlat(quantizer, dim, nlist, mt)
  index.train(Xf)
  index.add(Xf)
  build_time = time.time() - start

  index.nprobe = 10
  start = time.time()
  _, I = index.search(Qf, k)
  query_ms = (time.time() - start) * 1000 / len(Q)

  return build_time, query_ms, recall_at_k(I, gt, k)

In [6]:
def run_hnsw(X, Q, gt, k, metric):
  dim = X.shape[1]
  space = 'l2' if metric == 'euclidean' else 'cosine'
  p = hnswlib.Index(space=space, dim=dim)
  p.set_num_threads(1)  # supaya adil, sama-sama 1 thread

  start = time.time()
  p.init_index(max_elements=len(X), ef_construction=100, M=16)
  p.add_items(X)
  build_time = time.time() - start

  p.set_ef(50)
  start = time.time()
  labels, _ = p.knn_query(Q, k=k)
  query_ms = (time.time() - start) * 1000 / len(Q)

  return build_time, query_ms, recall_at_k(labels, gt, k)

In [7]:
k = 10
n_query = 100
rows = []

for n in [1_000, 1_000_000]:
  for dim in [2, 5]:
    np.random.seed(42)
    X = np.random.random((n, dim)).astype('float32')
    Q = np.random.random((n_query, dim)).astype('float32')

    for metric in ['euclidean', 'cosine']:
      gt = exact_nn(X, Q, k, metric)

      for name, fn in [('ANNOY', run_annoy), ('FAISS', run_faiss), ('HNSW', run_hnsw)]:
        b, q, r = fn(X, Q, gt, k, metric)
        rows.append({
          'Data/Dim': f'{n:,}/{dim}D',
          'Metric': metric,
          'Algo': name,
          'Build (s)': round(b, 3),
          'Query (ms)': round(q, 4),
          'Recall': round(r, 3),
        })
        print('selesai:', n, dim, metric, name)

df_res = pd.DataFrame(rows)

selesai: 1000 2 euclidean ANNOY
selesai: 1000 2 euclidean FAISS
selesai: 1000 2 euclidean HNSW
selesai: 1000 2 cosine ANNOY
selesai: 1000 2 cosine FAISS
selesai: 1000 2 cosine HNSW
selesai: 1000 5 euclidean ANNOY
selesai: 1000 5 euclidean FAISS
selesai: 1000 5 euclidean HNSW
selesai: 1000 5 cosine ANNOY
selesai: 1000 5 cosine FAISS
selesai: 1000 5 cosine HNSW
selesai: 1000000 2 euclidean ANNOY
selesai: 1000000 2 euclidean FAISS
selesai: 1000000 2 euclidean HNSW
selesai: 1000000 2 cosine ANNOY
selesai: 1000000 2 cosine FAISS
selesai: 1000000 2 cosine HNSW
selesai: 1000000 5 euclidean ANNOY
selesai: 1000000 5 euclidean FAISS
selesai: 1000000 5 euclidean HNSW
selesai: 1000000 5 cosine ANNOY
selesai: 1000000 5 cosine FAISS
selesai: 1000000 5 cosine HNSW


In [8]:
for col in ['Build (s)', 'Query (ms)', 'Recall']:
  print(f'\n=== {col} ===')
  print(df_res.pivot_table(index=['Data/Dim', 'Metric'], columns='Algo', values=col))


=== Build (s) ===
Algo                     ANNOY  FAISS     HNSW
Data/Dim     Metric                           
1,000,000/2D cosine     91.460  1.359  125.582
             euclidean  32.041  0.650   84.269
1,000,000/5D cosine     29.202  0.320  119.175
             euclidean  22.877  0.234  126.835
1,000/2D     cosine      0.077  0.117    0.051
             euclidean   0.041  0.003    0.070
1,000/5D     cosine      0.040  0.001    0.063
             euclidean   0.027  0.001    0.055

=== Query (ms) ===
Algo                     ANNOY   FAISS    HNSW
Data/Dim     Metric                           
1,000,000/2D cosine     0.0695  1.5878  0.0553
             euclidean  0.0702  1.1569  0.0305
1,000,000/5D cosine     0.0606  1.0093  0.0484
             euclidean  0.0599  0.6660  0.1019
1,000/2D     cosine     0.0308  0.0230  0.0159
             euclidean  0.0313  0.0250  0.0192
1,000/5D     cosine     0.0371  0.0124  0.0308
             euclidean  0.0330  0.0493  0.0253

=== Recall ===
Algo 